# Entrega Final — Dashboard sobre el Modelo Estrella

Responde las **12 preguntas de negocio del Paso 1** usando **solo el modelo estrella** que cargó `01_carga_dw.ipynb` en MySQL (no el CSV original).

La idea es:

1. Leer de MySQL las 5 tablas del modelo (`FACT_COMPETENCIA` y sus 4 dimensiones).
2. Unirlas en pandas en una sola tabla de partidos.
3. Para cada pregunta: filtrar, agrupar, calcular el indicador, mostrar la tabla y hacer un gráfico.

Los gráficos se guardan también como imagen en la carpeta `graficos/`.

**Recordatorio:** ejecutar las celdas en orden con **Shift + Enter**, o *Run → Run All Cells*.

## 0. Preparación

`matplotlib` es la librería de gráficos. pandas la usa por dentro cuando hacemos `tabla.plot(...)`.

In [ ]:
import os
from getpass import getpass

import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine, URL

SERVIDOR = "localhost"
PUERTO = 3306
USUARIO = "root"
BASE_DE_DATOS = "dw_competencia_futbol"

os.makedirs("graficos", exist_ok=True)   # crea la carpeta si no existe

### 0.1 Leer el modelo estrella desde MySQL

`pd.read_sql("SELECT ...", motor)` trae el resultado de la consulta como DataFrame.

In [ ]:
contrasena = os.environ.get("MYSQL_PASSWORD") or getpass("Contraseña de MySQL: ")
url = URL.create("mysql+mysqlconnector", username=USUARIO, password=contrasena,
                 host=SERVIDOR, port=PUERTO, database=BASE_DE_DATOS)
motor = create_engine(url)

fact = pd.read_sql("SELECT * FROM FACT_COMPETENCIA", motor)
dim_tiempo = pd.read_sql("SELECT * FROM DIM_TIEMPO", motor, parse_dates=["fecha"])
dim_equipo = pd.read_sql("SELECT * FROM DIM_EQUIPO", motor)
dim_division = pd.read_sql("SELECT * FROM DIM_DIVISION", motor)
dim_pais = pd.read_sql("SELECT * FROM DIM_PAIS", motor)

print("FACT_COMPETENCIA:", len(fact), "filas")

MySQL devuelve las columnas `DECIMAL` (el Elo) como un tipo especial de Python. Las pasamos a número decimal común (`float`) para poder hacer cuentas. Lo mismo con las tarjetas, que pueden tener vacíos.

In [ ]:
for columna in ["elo_local", "elo_visitante", "amarillas_local", "amarillas_visitante",
                "rojas_local", "rojas_visitante"]:
    fact[columna] = fact[columna].astype(float)

### 0.2 Unir los hechos con las dimensiones

Con `.merge` (el `JOIN` de pandas) le agregamos a cada partido su fecha, temporada, liga, país y el nombre de los dos equipos. El resultado es la tabla `partidos`: **una fila por partido**.

In [ ]:
partidos = fact.merge(dim_tiempo, on="id_tiempo")
partidos = partidos.merge(dim_division, on="id_division")
partidos = partidos.merge(dim_pais, on="id_pais")

local = dim_equipo.rename(columns={"id_equipo": "id_equipo_local", "nombre_equipo": "equipo_local"})
partidos = partidos.merge(local, on="id_equipo_local")
visitante = dim_equipo.rename(columns={"id_equipo": "id_equipo_visitante", "nombre_equipo": "equipo_visitante"})
partidos = partidos.merge(visitante, on="id_equipo_visitante")

partidos["liga"] = partidos["codigo_division"] + " - " + partidos["nombre_liga"]
partidos.head()

### 0.3 Tabla de participaciones (un partido visto desde cada equipo)

Muchas preguntas son "por equipo" (goles a favor, % de victorias…). Pero en `partidos` cada equipo aparece a veces como local y a veces como visitante.

Para simplificar, armamos la tabla `participacion`, donde **cada partido aparece dos veces**: una desde el punto de vista del local y otra desde el del visitante. Así, "goles a favor de un equipo" es siempre la columna `goles_favor`, sin importar dónde jugó. (Es lo mismo que la vista `V_PARTICIPACION` de la versión SQL.)

In [ ]:
columnas_comunes = ["fecha", "anio", "temporada", "codigo_division", "liga", "codigo_pais", "nombre_pais"]

desde_local = partidos[columnas_comunes].copy()
desde_local["equipo"] = partidos["equipo_local"]
desde_local["rival"] = partidos["equipo_visitante"]
desde_local["condicion"] = "Local"
desde_local["goles_favor"] = partidos["goles_local"]
desde_local["goles_contra"] = partidos["goles_visitante"]
desde_local["puntos"] = partidos["puntos_local"]
desde_local["victoria"] = partidos["victoria_local"]
desde_local["empate"] = partidos["empate"]
desde_local["derrota"] = partidos["victoria_visitante"]
desde_local["elo"] = partidos["elo_local"]
desde_local["amarillas"] = partidos["amarillas_local"]
desde_local["rojas"] = partidos["rojas_local"]

desde_visitante = partidos[columnas_comunes].copy()
desde_visitante["equipo"] = partidos["equipo_visitante"]
desde_visitante["rival"] = partidos["equipo_local"]
desde_visitante["condicion"] = "Visitante"
desde_visitante["goles_favor"] = partidos["goles_visitante"]
desde_visitante["goles_contra"] = partidos["goles_local"]
desde_visitante["puntos"] = partidos["puntos_visitante"]
desde_visitante["victoria"] = partidos["victoria_visitante"]
desde_visitante["empate"] = partidos["empate"]
desde_visitante["derrota"] = partidos["victoria_local"]
desde_visitante["elo"] = partidos["elo_visitante"]
desde_visitante["amarillas"] = partidos["amarillas_visitante"]
desde_visitante["rojas"] = partidos["rojas_visitante"]

participacion = pd.concat([desde_local, desde_visitante], ignore_index=True)
print("Participaciones:", len(participacion), "(= 2 x", len(partidos), "partidos)")

### 0.4 Fechas de referencia y función para graficar

Las preguntas que dicen "últimos 5 años" se cuentan desde **el último partido cargado**, no desde hoy.

La función `grafico_barras` la usamos en casi todas las preguntas: recibe la tabla, qué columna va en las etiquetas y cuál en las barras, dibuja un gráfico de barras horizontales y lo guarda como imagen. Así no repetimos el mismo código 12 veces.

In [ ]:
fecha_max = partidos["fecha"].max()
hace_5_anios = fecha_max - pd.DateOffset(years=5)
hace_3_anios = fecha_max - pd.DateOffset(years=3)
print("Último partido cargado:", fecha_max.date())
print("Ventana de 5 años desde:", hace_5_anios.date())
print("Ventana de 3 años desde:", hace_3_anios.date())

AZUL = "#2a78d6"


def grafico_barras(tabla, etiqueta, valor, titulo, texto_eje, archivo):
    # Se invierte el orden para que el primero de la tabla quede arriba
    datos = tabla.iloc[::-1]
    alto = max(3, 0.35 * len(datos) + 1)
    fig, ax = plt.subplots(figsize=(9, alto))
    barras = ax.barh(datos[etiqueta].astype(str), datos[valor], color=AZUL)
    ax.bar_label(barras, padding=3, fontsize=8)
    ax.set_title(titulo, loc="left")
    ax.set_xlabel(texto_eje)
    ax.spines[["top", "right"]].set_visible(False)
    ax.margins(x=0.1)
    fig.tight_layout()
    fig.savefig("graficos/" + archivo, dpi=120)
    plt.show()

---
## Pregunta 1 — ¿Cuánto mejoró un equipo específico en 5 años? (ejemplo: Nottingham Forest)

**Indicador:** Evolución de Elo · **Perspectivas:** Equipo, Tiempo

Para cambiar de equipo, alcanza con cambiar el nombre en `EQUIPO` (tiene que estar escrito como en `DIM_EQUIPO`).

`.groupby("anio")` agrupa las filas por año (como `GROUP BY` en SQL) y `.agg(...)` calcula en cada grupo el promedio (`mean`), mínimo, máximo y cantidad (`count`).

In [ ]:
EQUIPO = "Nottm Forest"

elo_equipo = participacion[
    (participacion["equipo"] == EQUIPO)
    & participacion["elo"].notna()
    & (participacion["fecha"] >= hace_5_anios)
].sort_values("fecha")

por_anio = elo_equipo.groupby("anio").agg(
    elo_promedio=("elo", "mean"),
    elo_minimo=("elo", "min"),
    elo_maximo=("elo", "max"),
    partidos=("elo", "count"),
).round(1).reset_index()
por_anio

Evolución: Elo del **primer** partido de la ventana contra Elo del **último** (`.iloc[0]` es la primera fila y `.iloc[-1]` la última).

In [ ]:
primero = elo_equipo.iloc[0]
ultimo = elo_equipo.iloc[-1]
variacion = ultimo["elo"] - primero["elo"]

print("Elo inicial:", primero["elo"], "el", primero["fecha"].date(), "en", primero["liga"])
print("Elo final:  ", ultimo["elo"], "el", ultimo["fecha"].date(), "en", ultimo["liga"])
print("Variación:  ", round(variacion, 1), "puntos (", round(100 * variacion / primero["elo"], 1), "% )")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.plot(por_anio["anio"], por_anio["elo_promedio"], marker="o", color=AZUL, linewidth=2)
for x, y in zip(por_anio["anio"], por_anio["elo_promedio"]):
    ax.annotate(str(round(y)), (x, y), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=8)
ax.set_xticks(por_anio["anio"])
ax.set_title("Nottingham Forest: Elo promedio por año", loc="left")
ax.set_ylabel("Elo promedio")
ax.spines[["top", "right"]].set_visible(False)
ax.margins(y=0.2)
fig.tight_layout()
fig.savefig("graficos/pregunta_01.png", dpi=120)
plt.show()

**Respuesta:** Nottingham Forest pasó de 1466 puntos de Elo (septiembre de 2021, en Championship) a 1780 (agosto de 2026): **+314 puntos, un 21,4 % de crecimiento**. El mayor salto fue entre 2024 y 2025.

---
## Pregunta 2 — ¿Qué equipo tiende a ganar más contra otro según su historial? (clásicos)

**Indicador:** % de victorias en enfrentamientos directos · **Perspectivas:** Equipo, Rival

Para cada clásico filtramos los partidos donde `equipo` es el primero y `rival` el segundo (incluye los partidos de local y de visitante). Usamos un `for` que recorre la lista de clásicos y va agregando una fila por clásico.

In [ ]:
clasicos = [
    ("Real Madrid", "Barcelona"),
    ("Arsenal", "Tottenham"),
    ("Sevilla", "Betis"),
    ("Man United", "Liverpool"),
    ("Celtic", "Rangers"),
    ("Inter", "Milan"),
]

filas = []
for equipo_a, equipo_b in clasicos:
    cruces = participacion[(participacion["equipo"] == equipo_a) & (participacion["rival"] == equipo_b)]
    jugados = len(cruces)
    filas.append({
        "clasico": equipo_a + " vs " + equipo_b,
        "partidos": jugados,
        "gana_a": cruces["victoria"].sum(),
        "empates": cruces["empate"].sum(),
        "gana_b": cruces["derrota"].sum(),
        "pct_gana_a": round(100 * cruces["victoria"].sum() / jugados, 1),
        "pct_empate": round(100 * cruces["empate"].sum() / jugados, 1),
        "pct_gana_b": round(100 * cruces["derrota"].sum() / jugados, 1),
    })

tabla_clasicos = pd.DataFrame(filas).sort_values("partidos", ascending=False)
tabla_clasicos

Gráfico de barras **apiladas** (`stacked=True`): cada barra suma 100 % y se ve qué parte ganó cada uno y qué parte fue empate.

In [ ]:
grafico = tabla_clasicos.iloc[::-1].plot(
    kind="barh", x="clasico", y=["pct_gana_a", "pct_empate", "pct_gana_b"], stacked=True,
    color=[AZUL, "#898781", "#eb6834"], figsize=(9, 4), width=0.6,
)
grafico.legend(["Gana el primero", "Empate", "Gana el segundo"], loc="lower center",
               bbox_to_anchor=(0.5, 1.02), ncol=3, frameon=False)
for barras in grafico.containers:
    grafico.bar_label(barras, fmt="%.0f%%", label_type="center", color="white", fontsize=8)
grafico.set_title("Resultados de los clásicos (% de partidos)", loc="left", pad=30)
grafico.set_xlabel("% de partidos")
grafico.set_ylabel("")
grafico.set_xlim(0, 100)
grafico.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("graficos/pregunta_02.png", dpi=120)
plt.show()

**Respuesta:** Arsenal domina a Tottenham (gana el 47 % contra el 20 %) y Sevilla a Betis (43 % contra 18 %); Celtic a Rangers (47 % contra 31 %), Man United a Liverpool (44 % contra 35 %) e Inter a Milan (44 % contra 36 %). En el clásico español gana más Barcelona (46 % contra 33 %).

---
## Pregunta 3 — ¿Existe ventaja de local?

**Indicadores:** Ventaja de localía, Promedio de goles a favor / en contra · **Perspectivas:** División

Como `victoria_local` vale 1 o 0, su **promedio** es directamente la proporción de partidos que ganó el local (multiplicado por 100 da el porcentaje).

In [ ]:
print("Partidos:", len(partidos))
print("% gana local:     ", round(100 * partidos["victoria_local"].mean(), 1))
print("% empate:         ", round(100 * partidos["empate"].mean(), 1))
print("% gana visitante: ", round(100 * partidos["victoria_visitante"].mean(), 1))
print("Goles del local por partido:    ", round(partidos["goles_local"].mean(), 2))
print("Goles del visitante por partido:", round(partidos["goles_visitante"].mean(), 2))

Por liga: la ventaja es la diferencia, en puntos porcentuales, entre el % de victorias del local y el del visitante.

In [ ]:
por_liga = partidos.groupby("liga").agg(
    partidos=("liga", "count"),
    pct_local=("victoria_local", "mean"),
    pct_visitante=("victoria_visitante", "mean"),
    goles_local=("goles_local", "mean"),
    goles_visitante=("goles_visitante", "mean"),
).reset_index()

por_liga["pct_local"] = 100 * por_liga["pct_local"]
por_liga["pct_visitante"] = 100 * por_liga["pct_visitante"]
por_liga["ventaja_pp"] = por_liga["pct_local"] - por_liga["pct_visitante"]
por_liga["dif_goles"] = por_liga["goles_local"] - por_liga["goles_visitante"]

ventaja_local = por_liga[["liga", "partidos", "pct_local", "pct_visitante", "ventaja_pp", "dif_goles"]]
ventaja_local = ventaja_local.sort_values("ventaja_pp", ascending=False).round(2)
ventaja_local

In [ ]:
grafico_barras(ventaja_local, "liga", "ventaja_pp",
               "Ventaja de localía por liga (% victorias local − % victorias visitante)",
               "puntos porcentuales", "pregunta_03.png")

**Respuesta:** Sí. En total el local gana el **45,0 %** de los partidos contra el **28,1 %** del visitante (+16,9 puntos porcentuales) y convierte 1,50 goles contra 1,14. La ventaja es mayor en La Liga (19,5 pp) y menor en la Premiership escocesa (10,6 pp).

---
## Pregunta 4 — ¿Qué liga es más pareja?

**Indicador:** Paridad de la liga · **Perspectivas:** División, Tiempo

Medimos la **brecha de Elo** entre los dos equipos de cada partido (`elo_local − elo_visitante`) y calculamos su **desvío estándar** en cada liga. Si el desvío es chico, los partidos suelen ser entre equipos de nivel parecido: la liga es **más pareja**.

Solo se consideran las ligas con Elo en al menos el 80 % de sus partidos.

In [ ]:
partidos["brecha_elo"] = partidos["elo_local"] - partidos["elo_visitante"]
partidos["brecha_elo_abs"] = partidos["brecha_elo"].abs()

paridad = partidos.groupby("liga").agg(
    partidos=("liga", "count"),
    partidos_con_elo=("brecha_elo", "count"),      # count no cuenta los vacíos
    desvio_brecha_elo=("brecha_elo", "std"),
    brecha_elo_media=("brecha_elo_abs", "mean"),
    pct_empates=("empate", "mean"),
).reset_index()
paridad["pct_empates"] = 100 * paridad["pct_empates"]

paridad = paridad[paridad["partidos_con_elo"] >= 0.8 * paridad["partidos"]]
paridad = paridad.sort_values("desvio_brecha_elo").round(1)
paridad

In [ ]:
grafico_barras(paridad, "liga", "desvio_brecha_elo",
               "Paridad: desvío de la brecha de Elo entre rivales (menor = más pareja)",
               "desvío estándar de (Elo local − Elo visitante)", "pregunta_04.png")

**Respuesta:** Las segundas divisiones son las más parejas (Segunda División, Ligue 2 y Serie B: desvío de ~80-85 puntos de Elo y ~30 % de empates). Entre las primeras divisiones la más pareja es la Ligue 1; las menos parejas son la Eredivisie, la Premiership escocesa y la Primeira Liga.

---
## Pregunta 5 — ¿Qué liga acumula más tarjetas?

**Indicador:** Promedio de tarjetas amarillas y rojas · **Perspectivas:** División, País

Se promedian **solo los partidos con dato de tarjetas** (los vacíos no cuentan como 0: pandas los saltea al calcular el promedio). Solo ligas con al menos 1000 partidos con dato.

In [ ]:
partidos["amarillas_total"] = partidos["amarillas_local"] + partidos["amarillas_visitante"]
partidos["rojas_total"] = partidos["rojas_local"] + partidos["rojas_visitante"]

tarjetas_liga = partidos.groupby(["liga", "nombre_pais"]).agg(
    partidos_con_dato=("amarillas_local", "count"),
    amarillas_x_partido=("amarillas_total", "mean"),
    rojas_x_partido=("rojas_total", "mean"),
).reset_index()

tarjetas_liga = tarjetas_liga[tarjetas_liga["partidos_con_dato"] >= 1000]
tarjetas_liga = tarjetas_liga.sort_values("amarillas_x_partido", ascending=False).round(3)
tarjetas_liga

In [ ]:
grafico_barras(tarjetas_liga, "liga", "amarillas_x_partido",
               "Tarjetas amarillas por partido, por liga", "amarillas por partido", "pregunta_05.png")

**Respuesta:** La Primeira Liga (5,09 amarillas por partido), La Liga (5,02) y la Segunda División española (5,00). Portugal y España también encabezan las rojas (~0,29 por partido). Las ligas inglesas y la Eredivisie son las que menos acumulan (~3 amarillas).

---
## Pregunta 6 — ¿Qué equipos tienen mejor estadística de visitante? (últimos 5 años)

**Indicadores:** % de victorias y Promedio de goles a favor como visitante · **Perspectivas:** Equipo, División

Solo equipos con al menos 60 partidos de visitante en la ventana. `.head(15)` se queda con los primeros 15.

In [ ]:
de_visitante = participacion[
    (participacion["condicion"] == "Visitante") & (participacion["fecha"] >= hace_5_anios)
]

visitantes = de_visitante.groupby("equipo").agg(
    divisiones=("codigo_division", "unique"),
    partidos_visitante=("equipo", "count"),
    pct_victorias=("victoria", "mean"),
    goles_favor_prom=("goles_favor", "mean"),
    goles_contra_prom=("goles_contra", "mean"),
).reset_index()
visitantes["pct_victorias"] = 100 * visitantes["pct_victorias"]

mejores_visitantes = visitantes[visitantes["partidos_visitante"] >= 60]
mejores_visitantes = mejores_visitantes.sort_values("pct_victorias", ascending=False).head(15).round(2)
mejores_visitantes

In [ ]:
grafico_barras(mejores_visitantes, "equipo", "pct_victorias",
               "Top 15 visitantes: % de victorias fuera de casa (últimos 5 años)",
               "% de victorias como visitante", "pregunta_06.png")

**Respuesta:** En los últimos 5 años, Porto (gana el 71,8 % de sus partidos como visitante), PSV (70,9 %) y Celtic (70,7 %). Entre las cinco grandes ligas, Bayern Munich (65,5 %), Paris SG y Barcelona.